In [67]:
# import statements
import pandas as pd
import requests
import json

import matplotlib.pyplot as plt


## Data Handling

In [68]:
#call data and clean (remove null and duplicate values)
df = pd.read_csv('agriculture_dataset.csv')
df=df.drop_duplicates()

missing_codes = ["--", "", " ", "nan", "NaN", "None"] #account for different possible null combos


for col in df.columns:
    if df[col].dtype == object:  # only clean string/object columns
        df[col] = df[col].str.strip().replace(missing_codes, pd.NA)
    else:
        df[col] = df[col].replace(missing_codes, pd.NA) 

print(df.columns)
df.head()

Index(['High_Resolution_RGB', 'Multispectral_Images', 'Thermal_Images',
       'Temporal_Images', 'Spatial_Resolution', 'GPS_Coordinates',
       'Field_Boundaries', 'Elevation_Data', 'Canopy_Coverage', 'NDVI', 'SAVI',
       'Chlorophyll_Content', 'Leaf_Area_Index', 'Crop_Stress_Indicator',
       'Temperature', 'Humidity', 'Rainfall', 'Wind_Speed', 'Soil_Moisture',
       'Soil_pH', 'Organic_Matter', 'Pest_Hotspots', 'Weed_Coverage',
       'Pest_Damage', 'Crop_Growth_Stage', 'Expected_Yield', 'Crop_Type',
       'Ground_Truth_Segmentation', 'Bounding_Boxes', 'Water_Flow',
       'Drainage_Features', 'Crop_Health_Label'],
      dtype='object')


,High_Resolution_RGB,Multispectral_Images,Thermal_Images,Temporal_Images,Spatial_Resolution,GPS_Coordinates,Field_Boundaries,Elevation_Data,Canopy_Coverage,NDVI,...,Weed_Coverage,Pest_Damage,Crop_Growth_Stage,Expected_Yield,Crop_Type,Ground_Truth_Segmentation,Bounding_Boxes,Water_Flow,Drainage_Features,Crop_Health_Label
0,0,0,0,0,0.667324,201538,3,28.207634,8.046926,0.676945,...,1.922274,84,2,2540.784327,Wheat,1,5,41.771884,0,1
1,1,1,0,0,1.459000,215854,3,82.335147,147.512332,0.414781,...,4.851381,56,3,3227.617025,Wheat,0,1,27.564635,0,1
2,0,0,0,0,0.500442,890802,3,83.865629,30.246527,0.723610,...,5.974859,38,1,4609.938146,Maize,1,8,29.510836,0,1
3,0,0,0,0,1.865161,605584,3,20.747905,6.857820,0.405611,...,2.100598,27,2,1409.716754,Maize,0,1,34.822855,0,0
4,0,1,1,1,1.392331,871732,3,22.588815,26.168558,0.465992,...,3.025669,84,4,3905.312588,Rice,0,2,15.493255,1,0


In [77]:
#clean dataset/create subsets
#drop pred value
df_sub=df.drop(columns='Crop_Health_Label', axis=1)

#one hot encoding to turn category into numerical value
df_sub = pd.get_dummies(df_sub, columns=['Crop_Type'], dtype=int)

#drop category and binary columns and crop health for pred
columnsToDrop=['High_Resolution_RGB','Multispectral_Images', 'Thermal_Images',
       'Temporal_Images','Field_Boundaries', 'Pest_Hotspots','Crop_Growth_Stage','Crop_Type_Maize','Crop_Type_Rice','Crop_Type_Wheat', 'Ground_Truth_Segmentation', 'Bounding_Boxes',
       'Drainage_Features']
df_sub_drop=df_sub.drop(columns=columnsToDrop, axis=1)

df_sub.head()

,High_Resolution_RGB,Multispectral_Images,Thermal_Images,Temporal_Images,Spatial_Resolution,GPS_Coordinates,Field_Boundaries,Elevation_Data,Canopy_Coverage,NDVI,...,Pest_Damage,Crop_Growth_Stage,Expected_Yield,Ground_Truth_Segmentation,Bounding_Boxes,Water_Flow,Drainage_Features,Crop_Type_Maize,Crop_Type_Rice,Crop_Type_Wheat
0,0,0,0,0,0.667324,201538,3,28.207634,8.046926,0.676945,...,84,2,2540.784327,1,5,41.771884,0,0,0,1
1,1,1,0,0,1.459000,215854,3,82.335147,147.512332,0.414781,...,56,3,3227.617025,0,1,27.564635,0,0,0,1
2,0,0,0,0,0.500442,890802,3,83.865629,30.246527,0.723610,...,38,1,4609.938146,1,8,29.510836,0,1,0,0
3,0,0,0,0,1.865161,605584,3,20.747905,6.857820,0.405611,...,27,2,1409.716754,0,1,34.822855,0,1,0,0
4,0,1,1,1,1.392331,871732,3,22.588815,26.168558,0.465992,...,84,4,3905.312588,0,2,15.493255,1,0,1,0


In [70]:
#scale the sub data
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
#scale numerical columns
df_sub_scaled=pd.DataFrame(scaler.fit_transform(df_sub_drop), columns=df_sub_drop.columns)

df_sub_scaled.head()

,Spatial_Resolution,GPS_Coordinates,Elevation_Data,Canopy_Coverage,NDVI,SAVI,Chlorophyll_Content,Leaf_Area_Index,Crop_Stress_Indicator,Temperature,Humidity,Rainfall,Wind_Speed,Soil_Moisture,Soil_pH,Organic_Matter,Weed_Coverage,Pest_Damage,Expected_Yield,Water_Flow
0,-1.067064,-1.340181,-1.029697,-0.839162,1.177837,0.628747,-0.242148,1.452764,1.021493,-0.072325,-0.739046,-0.648841,0.567093,1.267643,-0.829059,0.069272,-0.586518,1.195122,-0.580031,1.159373
1,0.518105,-1.285190,1.054578,1.954318,-0.570108,-0.617998,-0.799036,-0.444915,1.333064,0.536840,-0.902750,-0.300518,0.233291,-0.878763,0.550467,-0.120099,1.246409,0.225970,0.281020,0.175535
2,-1.401212,1.307425,1.113511,-0.394506,1.488973,0.925056,0.002005,-0.505892,0.329115,-0.294710,-0.217221,1.310860,-0.206330,0.972412,-0.843086,-0.819698,1.949441,-0.397055,2.013974,0.310308
3,1.331362,0.211844,-1.316947,-0.862980,-0.631249,-1.973178,-0.052851,-0.749552,-1.332593,-2.704025,-1.236336,-0.242198,2.763689,0.971982,0.190018,0.250038,-0.474930,-0.777793,-1.997999,0.678159
4,0.384615,1.234173,-1.246060,-0.476187,-0.228669,-1.082536,1.573734,-1.330815,1.021493,-0.497430,0.255079,-0.526728,0.202143,-1.356292,2.321364,0.773103,0.103946,1.195122,1.130617,-0.660395


In [79]:
#add the binary/category data back in
df_clean=pd.concat([df_sub_scaled,df_sub], axis=1)
df_clean.head()

,Spatial_Resolution,GPS_Coordinates,Elevation_Data,Canopy_Coverage,NDVI,SAVI,Chlorophyll_Content,Leaf_Area_Index,Crop_Stress_Indicator,Temperature,...,Pest_Damage,Crop_Growth_Stage,Expected_Yield,Ground_Truth_Segmentation,Bounding_Boxes,Water_Flow,Drainage_Features,Crop_Type_Maize,Crop_Type_Rice,Crop_Type_Wheat
0,-1.067064,-1.340181,-1.029697,-0.839162,1.177837,0.628747,-0.242148,1.452764,1.021493,-0.072325,...,84,2,2540.784327,1,5,41.771884,0,0,0,1
1,0.518105,-1.285190,1.054578,1.954318,-0.570108,-0.617998,-0.799036,-0.444915,1.333064,0.536840,...,56,3,3227.617025,0,1,27.564635,0,0,0,1
2,-1.401212,1.307425,1.113511,-0.394506,1.488973,0.925056,0.002005,-0.505892,0.329115,-0.294710,...,38,1,4609.938146,1,8,29.510836,0,1,0,0
3,1.331362,0.211844,-1.316947,-0.862980,-0.631249,-1.973178,-0.052851,-0.749552,-1.332593,-2.704025,...,27,2,1409.716754,0,1,34.822855,0,1,0,0
4,0.384615,1.234173,-1.246060,-0.476187,-0.228669,-1.082536,1.573734,-1.330815,1.021493,-0.497430,...,84,4,3905.312588,0,2,15.493255,1,0,1,0


## KNN Regressor

In [ ]:
def knn_reg(X_train, y_train, X_test, k, similarity_metric='cosine'):
    """
    Args:
    X_train: numpy array of shape (n_samples, n_features) - training data features
    y_train: numpy array of shape (n_samples,) - training data target values
    X_test: numpy array of shape (m_samples, n_features) - test data features
    k: int - number of nearest neighbors to consider
    similarity_metric: str - 'cosine' to determine the similarity measure
    
    Returns:
    y_pred: numpy array of shape (m_samples,) - predicted target values for the test data
    nearest_neighbors: list of lists - indices of the k nearest neighbors for each test point
    """
            